# 20 — In-Memory Filesystem (Amazon FAR style)

A hierarchical path-based store (think an in-memory `/`). Parts 1-3 build the core (concurrency at
Part 3); Parts 4-5 are stretch tasks (atomic move + errors, then parallel disk-usage). Fill stubs, run
each test cell, peek at the collapsed `ref_` solutions only after trying.

The tree is a plain dict: a **directory** is `dict[name -> child]`, a **file** is the tuple
`("file", content)`. The root is `{}`. `_split` (provided) turns `"/a/b"` into `["a", "b"]`.

---

## Part 1 — Core operations

Functions over a tree: `mkdir(tree, path)` (like `mkdir -p`), `write(tree, path, content)` (parent dir
must exist), `read(tree, path)`, `ls(tree, path) -> sorted child names`. Raise `ValueError` for invalid
paths (a file where a dir is expected, reading a directory, missing parent).

**Lock down:** `mkdir` creates intermediate dirs; `write` requires the parent to exist; `read`/`ls`
validate the node type.

In [ ]:
def _split(path):
    return [p for p in path.split("/") if p]


def mkdir(tree, path):
    raise NotImplementedError


def write(tree, path, content):
    raise NotImplementedError


def read(tree, path):
    raise NotImplementedError


def ls(tree, path):
    raise NotImplementedError

In [ ]:
# --- Part 1 tests ---
def _t1():
    fs = {}
    mkdir(fs, "/a/b")
    write(fs, "/a/b/f.txt", "hello")
    assert read(fs, "/a/b/f.txt") == "hello"
    assert ls(fs, "/a") == ["b"] and ls(fs, "/a/b") == ["f.txt"]
    write(fs, "/a/b/f.txt", "world")                  # overwrite
    assert read(fs, "/a/b/f.txt") == "world"
    for bad in ("/a/b",):                              # reading a directory
        try:
            read(fs, bad)
        except ValueError:
            pass
        else:
            raise AssertionError("expected ValueError")
    print("Part 1 OK")

_t1()

## Part 2 — Recursive remove & find

- `rm(tree, path)`: remove a file or an entire subtree (raise `ValueError` if it doesn't exist).
- `find(tree, path) -> list`: every **file path** at or under `path`, sorted.

**Lock down:** `find` recurses, sorting children for deterministic output; `find` on a single file
returns just that path.

In [ ]:
def rm(tree, path):
    raise NotImplementedError


def find(tree, path):
    raise NotImplementedError

In [ ]:
# --- Part 2 tests ---
def _t2():
    fs = {}
    mkdir(fs, "/a/b"); mkdir(fs, "/a/c")
    write(fs, "/a/b/x", "1"); write(fs, "/a/c/y", "2"); write(fs, "/a/z", "3")
    assert find(fs, "/a") == ["/a/b/x", "/a/c/y", "/a/z"]
    rm(fs, "/a/b")
    assert find(fs, "/a") == ["/a/c/y", "/a/z"]
    assert find(fs, "/a/z") == ["/a/z"]               # a single file
    print("Part 2 OK")

_t2()

## Part 3 — Thread-safe filesystem

`ConcurrentFS` wraps a tree with a lock and exposes `mkdir`/`write`/`read`/`ls`. Many threads create
directories and files at once without corrupting the tree.

**The invariant:** 8 threads each create their own `/uN` directory with 50 files leaves exactly 400
files. **Discuss:** even "different keys" mutate the same dict (resize races), so the structure needs a
lock; per-directory locks for finer granularity.

In [ ]:
import threading


class ConcurrentFS:
    def __init__(self):
        raise NotImplementedError

    def mkdir(self, path):
        raise NotImplementedError

    def write(self, path, content):
        raise NotImplementedError

    def read(self, path):
        raise NotImplementedError

    def ls(self, path):
        raise NotImplementedError

In [ ]:
# --- Part 3 tests ---
def _t3():
    fs = ConcurrentFS()

    def worker(t):
        fs.mkdir("/u%d" % t)
        for i in range(50):
            fs.write("/u%d/f%d" % (t, i), str(i))

    ts = [threading.Thread(target=worker, args=(t,)) for t in range(8)]
    for t in ts: t.start()
    for t in ts: t.join()
    assert sum(len(fs.ls("/u%d" % t)) for t in range(8)) == 400
    print("Part 3 OK")

_t3()

## Part 4 (stretch) — Atomic move & error handling

`move(tree, src, dst)`: move a file/subtree from `src` to `dst`. Validate **everything first** (src
exists, dst's parent is an existing dir, dst doesn't already exist) and only then mutate — so a failed
move leaves the tree untouched (atomic). Raise `FSError` on any problem.

**Discuss:** validate-then-commit for atomicity, rename semantics, and where you'd add
permissions/read-only checks.

In [ ]:
class FSError(Exception):
    pass


def move(tree, src, dst):
    raise NotImplementedError

In [ ]:
# --- Part 4 tests ---
def _t4():
    fs = {}
    mkdir(fs, "/a"); write(fs, "/a/x", "1")
    move(fs, "/a/x", "/a/y")
    assert read(fs, "/a/y") == "1"
    try:
        read(fs, "/a/x")                               # source is gone
    except ValueError:
        pass
    else:
        raise AssertionError("expected ValueError")
    try:
        move(fs, "/a/y", "/nope/z")                    # dst parent missing
    except FSError:
        pass
    else:
        raise AssertionError("expected FSError")
    assert read(fs, "/a/y") == "1"                     # unchanged after failed move
    try:
        move(fs, "/a/zzz", "/a/w")                     # src missing
    except FSError:
        pass
    else:
        raise AssertionError("expected FSError")
    print("Part 4 OK")

_t4()

## Part 5 (stretch) — Parallel disk usage

`parallel_du(subtrees, num_procs) -> list[int]`: compute total bytes (sum of file content lengths) for
each subtree in parallel with `ProcessPoolExecutor`; worker is `fs_workers.subtree_size`. (Subtrees are
plain dict/tuple nodes pulled from a filesystem — independent, so they parallelize cleanly.)

**Discuss:** subtrees are independent (embarrassingly parallel); GIL (the recursive walk is CPU-bound
-> processes); pickling the subtree to each worker.

In [ ]:
from concurrent.futures import ProcessPoolExecutor
import fs_workers


def parallel_du(subtrees, num_procs) -> list:
    raise NotImplementedError

In [ ]:
# --- Part 5 tests ---
def _t5():
    t1 = {"f": ("file", "hello"), "sub": {"g": ("file", "xy")}}   # 5 + 2 = 7
    t2 = {"h": ("file", "abcd")}                                  # 4
    assert parallel_du([t1, t2], 2) == [7, 4]
    print("Part 5 OK")

_t5()

## Likely follow-ups
- Symlinks / hardlinks; cycle detection; reference counting.
- Per-directory locks or a lock-free persistent (copy-on-write) tree for concurrent readers.
- Path normalization (`.`/`..`), permissions, quotas, and a real `du`/`find` with filters.

---
## Reference solutions (don't peek until you've tried)

In [ ]:
import threading
from concurrent.futures import ProcessPoolExecutor
import fs_workers


def _split(path):
    return [p for p in path.split("/") if p]


def ref_mkdir(tree, path):
    node = tree
    for p in _split(path):
        if p in node:
            if isinstance(node[p], tuple):
                raise ValueError("file in path: " + p)
        else:
            node[p] = {}
        node = node[p]


def _parent(tree, parts):
    node = tree
    for p in parts:
        if p not in node or isinstance(node[p], tuple):
            raise ValueError("no such directory: " + p)
        node = node[p]
    return node


def ref_write(tree, path, content):
    parts = _split(path)
    parent = _parent(tree, parts[:-1])
    name = parts[-1]
    if isinstance(parent.get(name), dict):
        raise ValueError("is a directory: " + name)
    parent[name] = ("file", content)


def ref_read(tree, path):
    parts = _split(path)
    parent = _parent(tree, parts[:-1])
    node = parent.get(parts[-1])
    if not isinstance(node, tuple):
        raise ValueError("not a file: " + path)
    return node[1]


def ref_ls(tree, path):
    node = tree
    for p in _split(path):
        if p not in node or isinstance(node[p], tuple):
            raise ValueError("not a directory: " + p)
        node = node[p]
    return sorted(node.keys())


def ref_rm(tree, path):
    parts = _split(path)
    parent = _parent(tree, parts[:-1])
    if parts[-1] not in parent:
        raise ValueError("no such path: " + path)
    del parent[parts[-1]]


def ref_find(tree, path):
    node = tree
    base = ""
    for p in _split(path):
        node = node[p]
        base += "/" + p
    out = []

    def rec(n, prefix):
        if isinstance(n, tuple):
            out.append(prefix)
            return
        for k in sorted(n):
            rec(n[k], prefix + "/" + k)

    rec(node, base)
    return sorted(out)


class RefConcurrentFS:
    def __init__(self):
        self.tree = {}
        self.lock = threading.Lock()

    def mkdir(self, path):
        with self.lock:
            ref_mkdir(self.tree, path)

    def write(self, path, content):
        with self.lock:
            ref_write(self.tree, path, content)

    def read(self, path):
        with self.lock:
            return ref_read(self.tree, path)

    def ls(self, path):
        with self.lock:
            return ref_ls(self.tree, path)


class FSError(Exception):
    pass


def ref_move(tree, src, dst):
    sparts, dparts = _split(src), _split(dst)
    try:
        sparent = _parent(tree, sparts[:-1])
        dparent = _parent(tree, dparts[:-1])
    except ValueError:
        raise FSError("bad directory in path")
    if sparts[-1] not in sparent:
        raise FSError("no such source: " + src)
    if dparts[-1] in dparent:
        raise FSError("destination exists: " + dst)
    dparent[dparts[-1]] = sparent.pop(sparts[-1])      # validated -> single atomic mutation


def ref_parallel_du(subtrees, num_procs):
    with ProcessPoolExecutor(max_workers=num_procs) as ex:
        return list(ex.map(fs_workers.subtree_size, subtrees))


_fs = {}
ref_mkdir(_fs, "/a/b"); ref_write(_fs, "/a/b/f", "hi")
assert ref_read(_fs, "/a/b/f") == "hi" and ref_ls(_fs, "/a") == ["b"]
assert ref_find(_fs, "/a") == ["/a/b/f"]
ref_move(_fs, "/a/b/f", "/a/g"); assert ref_read(_fs, "/a/g") == "hi"
try:
    ref_move(_fs, "/a/g", "/zzz/q")
except FSError:
    pass
else:
    raise AssertionError("expected FSError")
assert ref_parallel_du([{"x": ("file", "abc")}], 2) == [3]
print("reference solutions OK")